# Instalación de RAG Lab en Google Colab con GPU L4

Este notebook te guía **paso a paso** para configurar tu ambiente en Colab con:
- **GPU L4** (recomendado: 2.8x más rápido que T4, buen precio)
- **Google Drive** como almacenamiento persistente (memoria, caché, PDFs)
- El paquete **rag_lab** listo para entrenar

> **Tiempo estimado:** 10–15 minutos. Ejecuta las celdas en orden.
> No saltes ninguna; las verificaciones te avisan si algo falla.

---

## ANTES DE EMPEZAR

1. **Estructura de carpetas en Drive.** En [Google Drive web](https://drive.google.com):
   - Crea carpeta `rag_lab` en "Mi unidad".
   - Dentro, crea 4 subcarpetas vacías: `rag_lab_proyecto`, `rag_memory`, `hf_cache`, `pdfs`.
   - Sube `rag_lab_proyecto.zip` a la carpeta `rag_lab/`.

2. **Claves de API.** En Colab (icono 🔑 en la barra izquierda, bajo Tools):
   - Añade secreto `OPENAI_API_KEY` (tuya de OpenAI).
   - Añade secreto `VOYAGE_API_KEY` (Voyage AI, para sus embeddings/rerankers).
   - Añade secreto `ANTHROPIC_API_KEY` (tuya de Anthropic).

3. **GPU L4.** En el menú Runtime (arriba):
   - Runtime → Change runtime type → Hardware accelerator: **L4** → Save.

---

## Celda 1: Cambiar a GPU L4 (opcional, si aún no lo hiciste)

In [ ]:
# Este notebook ASUME que ya cambiaste a L4. Si aún no:
# 1. Menú Runtime (arriba) → Change runtime type
# 2. Hardware accelerator: L4
# 3. Click Save (se reinicia)
# 4. Vuelve aquí y ejecuta esta celda

import torch
print("Dispositivo detectado:")
if torch.cuda.is_available():
    print("  ✓ GPU disponible:", torch.cuda.get_device_name(0))
    print("  Memoria:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
else:
    print("  ✗ SIN GPU. Ve a Runtime > Change runtime type y elige L4.")
    print("  Luego reinicia este notebook (Runtime > Restart session).")

## Celda 2: Montar Google Drive

In [ ]:
from google.colab import drive
import os

print("Montando Google Drive...")
print("(Te pedirá permiso; haz click en el link, autoriza y copia el código)")
drive.mount("/content/drive", force_remount=True)
print("✓ Drive montado en /content/drive")

# Verifica que tu carpeta rag_lab existe
rag_lab_path = "/content/drive/MyDrive/rag_lab"
if os.path.exists(rag_lab_path):
    print(f"✓ Carpeta rag_lab encontrada en Drive")
    print(f"  Contenido: {os.listdir(rag_lab_path)}")
else:
    print(f"✗ CARPETA NO ENCONTRADA: {rag_lab_path}")
    print("  Crea la carpeta 'rag_lab' en tu Drive primero (véase Antes de empezar).")

## Celda 3: Descomprimir el ZIP

In [ ]:
import zipfile
import os

rag_lab_path = "/content/drive/MyDrive/rag_lab"
zip_path = f"{rag_lab_path}/rag_lab_proyecto.zip"
destino = rag_lab_path

print(f"Buscando ZIP en {zip_path}...")

if os.path.exists(zip_path):
    print("✓ ZIP encontrado. Descomprimiendo...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(destino)
    print("✓ Descompresión completa.")
    print(f"  Contenido de {rag_lab_path}:")
    for item in os.listdir(rag_lab_path):
        print(f"    - {item}")
elif os.path.exists(f"{rag_lab_path}/rag_lab_proyecto"):
    print("✓ Ya descomprimido (rag_lab_proyecto/ existe).")
else:
    print(f"✗ ZIP NO ENCONTRADO: {zip_path}")
    print("  Sube rag_lab_proyecto.zip a tu carpeta rag_lab/ en Drive.")

## Celda 4: Instalar dependencias (requirements.txt)

In [ ]:
import subprocess
import os

rag_lab_path = "/content/drive/MyDrive/rag_lab/rag_lab_proyecto"
req_path = f"{rag_lab_path}/requirements.txt"

print(f"Leyendo {req_path}...")
if os.path.exists(req_path):
    with open(req_path) as f:
        contenido = f.read()
    print("Primeras líneas de requirements.txt:")
    for linea in contenido.split('\n')[:15]:
        if linea.strip() and not linea.startswith('#'):
            print(f"  {linea}")
    print("  ...")
    
    print("\nInstalando dependencias. Esto tarda 3–5 minutos...")
    resultado = subprocess.run(
        ["pip", "install", "-q", "-r", req_path],
        capture_output=True, text=True
    )
    if resultado.returncode == 0:
        print("✓ Instalación completada sin errores.")
    else:
        print("⚠ Algunos warnings (normales):")
        if resultado.stderr:
            lineas = resultado.stderr.split('\n')[:5]
            for l in lineas:
                if l.strip(): print(f"  {l}")
else:
    print(f"✗ requirements.txt NO ENCONTRADO: {req_path}")

## Celda 5: Configurar variables de entorno

In [ ]:
import os
import pathlib

rag_lab_path = "/content/drive/MyDrive/rag_lab"

# Directorios clave (ANTES de importar rag_lab)
MEM_DIR = f"{rag_lab_path}/rag_memory"
HF_DIR = f"{rag_lab_path}/hf_cache"

# Crear si no existen
pathlib.Path(MEM_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(HF_DIR).mkdir(parents=True, exist_ok=True)

# Fijar en el entorno (CRÍTICO: antes de importar rag_lab)
os.environ["RAG_MEMORY_DIR"] = MEM_DIR
os.environ["HF_HOME"] = HF_DIR

print("Variables de entorno configuradas:")
print(f"  RAG_MEMORY_DIR = {os.environ['RAG_MEMORY_DIR']}")
print(f"  HF_HOME = {os.environ['HF_HOME']}")
print(f"\nContenido de rag_memory (debe estar vacío o tener memoria_unificada.json):")
print(f"  {os.listdir(MEM_DIR)}")
print(f"\nContenido de hf_cache (se llena al descargar modelos):")
print(f"  {os.listdir(HF_DIR) if os.listdir(HF_DIR) else '(vacío por ahora)'}")

## Celda 6: Cargar claves de API desde secretos

In [ ]:
import os
from google.colab import userdata

print("Cargando claves de API desde el gestor de secretos...")
print("(Si aparece un diálogo, haz click en 'Grant access')\n")

claves = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "VOYAGE_API_KEY", "HF_TOKEN", "COHERE_API_KEY"]
cargadas = {}

for clave in claves:
    try:
        valor = userdata.get(clave)
        if valor:
            os.environ[clave] = valor
            cargadas[clave] = True
            print(f"✓ {clave:20} cargada (primeros 10 chars: {valor[:10]}...)")
        else:
            print(f"  {clave:20} no está definida (opcional; algunas funciones fallarán sin ella)")
            cargadas[clave] = False
    except Exception as e:
        print(f"  {clave:20} no encontrada: {e}")
        cargadas[clave] = False

# Verificación crítica
if not cargadas.get("OPENAI_API_KEY") or not cargadas.get("ANTHROPIC_API_KEY"):
    print("\n⚠ ADVERTENCIA: falta OPENAI_API_KEY y/o ANTHROPIC_API_KEY.")
    print("  El entrenamiento real necesitará ambas.")
    print("  Puedes continuar para probar con DOBLES (sin coste), pero ajusta antes de entrenar.")
else:
    print("\n✓ Todas las claves cargadas.")

## Celda 7: Configurar sys.path (añadir el paquete a Python)

In [ ]:
import sys
import pathlib

rag_lab_path = "/content/drive/MyDrive/rag_lab/rag_lab_proyecto"

# Añadir a sys.path en orden de prioridad
sys.path.insert(0, rag_lab_path)           # paquete rag_lab
sys.path.insert(0, f"{rag_lab_path}/tests")  # tests (dobles)

print("sys.path actualizado:")
for i, p in enumerate(sys.path[:5]):
    print(f"  {i}: {p}")

print("\nEsto permite que Python encuentre 'from rag_lab import ...'")

## Celda 8: Importar el paquete y verificar

In [ ]:
print("Importando rag_lab y módulos clave...\n")

try:
    from rag_lab import CONFIG, MetaSupervisor, Plan
    print("✓ rag_lab.main importado")
    
    from rag_lab.memory import Memoria
    print("✓ rag_lab.memory importado")
    
    from rag_lab.fabricas import _dispositivo_hf
    print("✓ rag_lab.fabricas importado")
    
    from dobles import dobles_activos
    print("✓ dobles (para ensayos sin coste) importado")
    
    print("\nTodos los imports exitosos. Sistema listo.")
    
except ImportError as e:
    print(f"✗ ERROR de importación: {e}")
    print("  Verifica que:")
    print("    - El zip está descomprimido en rag_lab_proyecto/")
    print("    - requirements.txt fue instalado correctamente")
    print("    - sys.path apunta a la carpeta correcta")

## Celda 9: Información de configuración

In [ ]:
import os

print("=" * 70)
print("RESUMEN DE CONFIGURACIÓN")
print("=" * 70)

print("\nDIRECTORIOS:")
print(f"  Proyecto: /content/drive/MyDrive/rag_lab/rag_lab_proyecto")
print(f"  Memoria:  {os.environ.get('RAG_MEMORY_DIR', '(no configurado)')}")
print(f"  Caché HF: {os.environ.get('HF_HOME', '(no configurado)')}")

print("\nAPI KEYS CARGADAS:")
for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "VOYAGE_API_KEY", "HF_TOKEN", "COHERE_API_KEY"]:
    v = os.environ.get(k)
    estado = f"✓ (primeros 10: {v[:10]}...)" if v else "✗ no definida"
    print(f"  {k:20} {estado}")

print("\nGPU / DISPOSITIVO:")
try:
    from rag_lab.fabricas import _dispositivo_hf
    device = _dispositivo_hf()
    print(f"  Dispositivo HF: {device}")
except:
    print("  Dispositivo HF: (error al detectar)")

try:
    import torch
    if torch.cuda.is_available():
        print(f"  GPU PyTorch: {torch.cuda.get_device_name(0)}")
        print(f"  Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("  GPU PyTorch: (no disponible)")
except:
    pass

print("\n" + "=" * 70)
print("✓ INSTALACIÓN COMPLETADA")
print("=" * 70)
print("\nPróximo paso: Abre entrenar.ipynb o analizar.ipynb")

## Celda 10: (Opcional) Prueba rápida con DOBLES

Si quieres verificar que todo funciona **sin gastar API**, ejecuta una prueba rápida con
dobles (FakeLLM + FakeEmbeddings). Esto confirma que el setup es sólido antes de
entrenar de verdad.

In [ ]:
import pathlib
import tempfile

print("Ensayo rápido del sistema con dobles (sin coste, sin red)...")
print("Esto tarda ~30 segundos.\n")

try:
    from rag_lab import MetaSupervisor, Plan
    from rag_lab.memory import Memoria
    from dobles import dobles_activos
    
    mem = Memoria(pathlib.Path(tempfile.mkdtemp()) / "test.json")
    
    with dobles_activos():
        meta = MetaSupervisor(mem)
        print("Lanzando meta-supervisor con dobles (splitter + retriever, 1 run)...")
        stack = meta.ejecutar("(demo).pdf", plan=Plan(bancos=["splitter", "retriever"], n_runs=1))
    
    print("✓ Ejecución completada.")
    print("\nResultado:")
    print(f"  Confianza global: {stack.get('confianza_global')}")
    print(f"  Stack recomendado (primeras claves): {list(stack.get('stack_recomendado', {}).keys())[:3]}")
    print(f"  Memoria acumulada: {mem.resumen()}")
    print("\n✓ Sistema FUNCIONAL. Listo para entrenar.")
    
except Exception as e:
    print(f"✗ ERROR durante el ensayo: {e}")
    print("  Revisa los imports y la configuración arriba.")
    import traceback
    traceback.print_exc()

---

## Notas finales

- **Si algo falla:** mira el error, verifica que completaste todos los "Antes de empezar" y
  ejecuta la celda de nuevo. Si persiste, anota el traceback y consulta.
- **La primera vez descarga modelos:** cuando corres entrenar.ipynb por primera vez, se
  descargan los embeddings y rerankers locales (~500MB). Esto va a `hf_cache` en Drive
  y no se re-descarga.
- **Memoria persistente:** cada experimento se guarda en `rag_memory/memoria_unificada.json`.
  Si reconectas a Colab mañana, esa memoria sigue ahí.

**¡Listo para entrenar!** Abre `entrenar.ipynb` y sigue adelante.